In [6]:
# Desinstalar para limpiar
!pip uninstall -y torch torchtext torchdata


Found existing installation: torch 2.2.2
Uninstalling torch-2.2.2:
  Successfully uninstalled torch-2.2.2
Found existing installation: torchtext 0.17.2
Uninstalling torchtext-0.17.2:
  Successfully uninstalled torchtext-0.17.2
Found existing installation: torchdata 0.7.1
Uninstalling torchdata-0.7.1:
  Successfully uninstalled torchdata-0.7.1


In [7]:
# Instalar el trío compatible
!pip install torch==2.2.2 torchtext==0.17.2 torchdata==0.7.1 portalocker>=2.0.0

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchtune 0.6.1 requires torchdata==0.11.0, but you have torchdata 0.7.1 which is incompatible.
torchaudio 2.10.0+cu128 requires torch==2.10.0, but you have torch 2.2.2 which is incompatible.
torchvision 0.25.0+cu128 requires torch==2.10.0, but you have torch 2.2.2 which is incompatible.


In [2]:
!pip uninstall -y numpy

Found existing installation: numpy 2.0.2
Uninstalling numpy-2.0.2:
  Successfully uninstalled numpy-2.0.2


In [1]:
!pip install numpy==1.26.4

In [2]:
import torch
import torchtext
from torchtext.datasets import AG_NEWS
import portalocker

print(f"Versión Torch: {torch.__version__}")
print(f"Versión Torchtext: {torchtext.__version__}")

# Prueba rápida para ver si AG_NEWS responde
try:
    train_iter = iter(AG_NEWS(split="train"))
    print("¡Éxito! El dataset está listo.")
except Exception as e:
    print(f"Error al cargar dataset: {e}")

Versión Torch: 2.2.2+cu121
Versión Torchtext: 0.17.2+cpu
¡Éxito! El dataset está listo.


In [3]:
next(train_iter)

(3,
 "Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again.")

In [4]:
from torchtext.data.utils import get_tokenizer
from torchtext.vocab import build_vocab_from_iterator

tokenizador = get_tokenizer('spacy' , language="en_core_web_sm")
train_iter = AG_NEWS(split='train')
train_iter = iter(train_iter)

def yield_tokens(data_iter):

  for _, text in data_iter:

    yield tokenizador(text)



In [5]:
vocab = build_vocab_from_iterator(yield_tokens(train_iter) , specials=['<unk>'])
vocab.set_default_index(vocab['<unk>'])


In [6]:
vocab(tokenizador("Hello how are you ? I am a engineer system student"))

[18574, 446, 45, 191, 134, 310, 3191, 6, 4681, 279, 4369]

In [17]:
# Pipelines: transforman texto a IDs y etiquetas a base 0
text_pipeline = lambda x: vocab(tokenizador(x))
label_pipeline = lambda x: int(x) - 1

In [18]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [20]:
def collate_batch(batch):
    label_list, text_list, offsets = [], [], [0]
    for (_label, _text) in batch:
         label_list.append(label_pipeline(_label))
         processed_text = torch.tensor(text_pipeline(_text), dtype=torch.int64)
         text_list.append(processed_text)
         offsets.append(processed_text.size(0)) # Guardamos el largo de cada noticia

    label_list = torch.tensor(label_list, dtype=torch.int64)
    # Convertimos los largos en posiciones acumuladas (necesario para EmbeddingBag)
    offsets = torch.tensor(offsets[:-1]).cumsum(dim=0)
    text_list = torch.cat(text_list)
    return label_list.to(device), text_list.to(device), offsets.to(device)

In [21]:
from torch.utils.data import DataLoader


train_iter = AG_NEWS(split='train')
dataloader = DataLoader(train_iter, batch_size=10, shuffle=True, collate_fn=collate_batch)

In [22]:
from torch import nn
import torch.nn.functional as f

class ModelTextClassifier(nn.Module):
  def __init__(self, vocab_size, embed_dim , num_class) -> None:
    super().__init__()

    self.embedding = nn.EmbeddingBag(vocab_size, embed_dim)

    self.bnl = nn.BatchNorm1d(embed_dim)

    self.fc1 = nn.Linear(embed_dim, num_class)

  def forward(self, text , offsets):

    embedded = self.embedding(text , offsets)
    embedded_norm = self.bnl(embedded)

    embedded_activate = f.relu(embedded_norm)

    return self.fc1(embedded_activate)

In [23]:
train_iter = AG_NEWS(split='train')
num_class = len(set([label for (label , text ) in train_iter ]))
vocab_size = len(vocab)
embedding_size = 100

modelo = ModelTextClassifier(vocab_size=vocab_size,embed_dim=embedding_size,num_class=num_class).to(device)

In [24]:
vocab_size

110933

In [25]:
modelo

ModelTextClassifier(
  (embedding): EmbeddingBag(110933, 100, mode='mean')
  (bnl): BatchNorm1d(100, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (fc1): Linear(in_features=100, out_features=4, bias=True)
)

In [26]:
def count_parameters(model):
  return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'El modelo tiene {count_parameters(modelo):,} ')

El modelo tiene 11,093,904 


In [27]:
import time
def entrenamiento(dataloader, model, optimizer, criterion):
    model.train()
    total_acc, total_count = 0, 0
    log_interval = 500
    start_time = time.time()

    for idx, (label, text, offsets) in enumerate(dataloader):
        optimizer.zero_grad()
        predicted_label = model(text, offsets)
        loss = criterion(predicted_label, label)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 0.1) # Clip de gradientes
        optimizer.step()

        total_acc += (predicted_label.argmax(1) == label).sum().item()
        total_count += label.size(0)

        if idx % log_interval == 0 and idx > 0:
            elapsed = time.time() - start_time
            print(f'| época {epoch:3d} | {idx:5d}/{len(dataloader):5d} batches '
                  f'| precisión {total_acc/total_count:8.3f}')

    return total_acc / total_count

In [28]:
def evalua(dataloader, model, criterion):
    model.eval()
    total_acc, total_count = 0, 0

    with torch.no_grad():
        for idx, (label, text, offsets) in enumerate(dataloader):
            predicted_label = model(text, offsets)
            loss = criterion(predicted_label, label)
            total_acc += (predicted_label.argmax(1) == label).sum().item()
            total_count += label.size(0) # CORREGIDO: antes decía total_account

    return total_acc / total_count

In [29]:
epochs = 3
lr = 0.1
batch_size = 64

In [30]:
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(modelo.parameters(),lr = lr )

In [31]:
from torch.utils.data.dataset import random_split
from torchtext.data.functional import to_map_style_dataset

train_iter , test_iter = AG_NEWS()

x_train = to_map_style_dataset(train_iter)
x_test = to_map_style_dataset(test_iter)


num_train = int(len(x_train) * 0.95 )

split_train , split_valid = random_split(x_train , [num_train , len(x_train) - num_train])

train_dataloader = DataLoader(split_train, batch_size=batch_size, shuffle = True , collate_fn = collate_batch)
valid_dataloader = DataLoader(split_valid, batch_size=batch_size, shuffle = True , collate_fn = collate_batch)
test_dataloader =  DataLoader(x_test     , batch_size=batch_size, shuffle = True , collate_fn = collate_batch)


In [33]:
epochs = 10
best_valid_acc = 0

for epoch in range(1, epochs + 1):
    epoch_start_time = time.time()

    # CORREGIDO: Pasamos todos los argumentos necesarios
    train_acc = entrenamiento(train_dataloader, modelo, optimizer, criterion)
    valid_acc = evalua(valid_dataloader, modelo, criterion)

    if valid_acc > best_valid_acc:
        best_valid_acc = valid_acc
        # Sugerencia: Guardar el mejor modelo
        torch.save(modelo.state_dict(), 'mejor_modelo.pt')

    print('-' * 59)
    print(f'| fin de época {epoch:3d} | tiempo: {time.time() - epoch_start_time:5.2f}s | '
          f'precisión validación {valid_acc:8.3f} ')
    print('-' * 59)

| época   1 |   500/ 1782 batches | precisión    0.373
| época   1 |  1000/ 1782 batches | precisión    0.416
| época   1 |  1500/ 1782 batches | precisión    0.437
-----------------------------------------------------------
| fin de época   1 | tiempo: 46.56s | precisión validación    0.479 
-----------------------------------------------------------
| época   2 |   500/ 1782 batches | precisión    0.485
| época   2 |  1000/ 1782 batches | precisión    0.488
| época   2 |  1500/ 1782 batches | precisión    0.489
-----------------------------------------------------------
| fin de época   2 | tiempo: 45.29s | precisión validación    0.495 
-----------------------------------------------------------
| época   3 |   500/ 1782 batches | precisión    0.497
| época   3 |  1000/ 1782 batches | precisión    0.498
| época   3 |  1500/ 1782 batches | precisión    0.498
-----------------------------------------------------------
| fin de época   3 | tiempo: 45.90s | precisión validación    0.504